## Prolog


In [1]:
"""
Sistema Esperto in Prolog per la Logistica Amazon
Genera automaticamente una Knowledge Base da dati della Rete Bayesiana
"""

import pandas as pd
import os

def normalizza_valore(valore: str) -> str:
    """Normalizza i valori per la sintassi Prolog (lowercase, no spazi)."""
    if pd.isna(valore):
        return 'unknown'
    return str(valore).lower().strip().replace(' ', '_').replace('-', '_')

def genera_header() -> str:
    """Genera l'header del file Prolog."""
    return """% ============================================
% SISTEMA ESPERTO - Logistica Amazon
% Knowledge Base generata automaticamente da Python
% ============================================
%
% Struttura: consegna(ID, Meteo, Traffico, EtaAutista, Status)
%
% Regole di Business:
%   1. Sicurezza: annulla con maltempo + ritardo
%   2. Supporto: contatta autista giovane in traffico critico
%   3. Performance: segnala ritardi ingiustificati
%   4. Default: procedere normalmente
%
% ============================================

"""

def genera_fatti(df: pd.DataFrame) -> str:
    """Genera i fatti Prolog dalle prime 20 righe del DataFrame."""
    fatti = []
    fatti.append("% ============================================")
    fatti.append("% FATTI - Dati delle consegne")
    fatti.append("% consegna(ID, Meteo, Traffico, EtaAutista, Status)")
    fatti.append("% ============================================\n")
    
    for idx, row in df.head(20).iterrows():
        id_ordine = f"ord_{idx}"
        meteo = normalizza_valore(row.get('Weather_conditions', 'unknown'))
        traffico = normalizza_valore(row.get('Road_traffic_density', 'unknown'))
        eta = normalizza_valore(row.get('Age_Category', 'unknown'))
        status = normalizza_valore(row.get('Delivery_Status', 'unknown'))
        
        fatto = f"consegna({id_ordine}, {meteo}, {traffico}, {eta}, {status})."
        fatti.append(fatto)
    
    return '\n'.join(fatti)

def genera_regole() -> str:
    """Genera le regole di business del sistema esperto."""
    return """
% ============================================
% REGOLE DI BUSINESS - Sistema Esperto Logistica
% ============================================

% ------------------------------------------
% REGOLA 1: SICUREZZA
% Condizione: meteo estremo (stormy/sandstorms) E consegna in ritardo
% Azione: annullare e riprogrammare la consegna
% Motivazione: Troppo pericoloso guidare col maltempo se già in ritardo
% ------------------------------------------
azione(ID, annulla_e_riprogramma) :-
    consegna(ID, Meteo, _, _, ritardo),
    (Meteo = stormy ; Meteo = sandstorms),
    !.

% ------------------------------------------
% REGOLA 2: SUPPORTO
% Condizione: traffico critico (jam/high) E autista giovane
% Azione: contattare l'autista per fornire supporto
% Motivazione: Autista inesperto in traffico critico necessita supporto dalla centrale
% ------------------------------------------
azione(ID, contatta_autista) :-
    consegna(ID, _, Traffico, giovane, _),
    (Traffico = jam ; Traffico = high),
    !.

% ------------------------------------------
% REGOLA 3: PERFORMANCE
% Condizione: meteo buono (sunny) E consegna in ritardo
% Azione: segnalazione performance negativa
% Motivazione: Ritardo ingiustificato col bel tempo, flaggare driver
% ------------------------------------------
azione(ID, segnalazione_performance) :-
    consegna(ID, sunny, _, _, ritardo),
    !.

% ------------------------------------------
% REGOLA 4: DEFAULT
% Condizione: nessuna delle regole precedenti si applica
% Azione: procedere normalmente con la consegna
% ------------------------------------------
azione(ID, procedere) :-
    consegna(ID, _, _, _, _),
    \\+ azione_speciale(ID).

% Predicato ausiliario per verificare azioni speciali
azione_speciale(ID) :-
    consegna(ID, Meteo, _, _, ritardo),
    (Meteo = stormy ; Meteo = sandstorms).
azione_speciale(ID) :-
    consegna(ID, _, Traffico, giovane, _),
    (Traffico = jam ; Traffico = high).
azione_speciale(ID) :-
    consegna(ID, sunny, _, _, ritardo).

% ============================================
% QUERY DI UTILITA'
% ============================================

% Trova tutte le consegne da annullare per sicurezza
consegne_da_annullare(Lista) :-
    findall(ID, azione(ID, annulla_e_riprogramma), Lista).

% Trova tutti gli autisti da contattare
autisti_da_contattare(Lista) :-
    findall(ID, azione(ID, contatta_autista), Lista).

% Trova tutte le segnalazioni performance
segnalazioni_performance(Lista) :-
    findall(ID, azione(ID, segnalazione_performance), Lista).

% Report completo con dettagli
report_consegna(ID, Meteo, Traffico, Eta, Status, Azione) :-
    consegna(ID, Meteo, Traffico, Eta, Status),
    azione(ID, Azione).

% Conta azioni per tipo
conta_azioni(TipoAzione, Conteggio) :-
    findall(ID, azione(ID, TipoAzione), Lista),
    length(Lista, Conteggio).
"""

def stampa_istruzioni_swish():
    """Stampa le istruzioni per testare il file su SWISH Prolog."""
    print("""
╔══════════════════════════════════════════════════════════════════╗
║       ISTRUZIONI PER TESTARE SU SWISH PROLOG                    ║
╚══════════════════════════════════════════════════════════════════╝

📌 PASSO 1: Accedi a SWISH Prolog
   → https://swish.swi-prolog.org/

📌 PASSO 2: Carica la Knowledge Base
   → Clicca "File" → "New" → "Program"
   → Copia il contenuto di 'sistema_esperto.pl'

📌 PASSO 3: Esegui le Query nel pannello in basso

   ┌─────────────────────────────────────────────────────────────┐
   │ QUERY PRINCIPALE                                            │
   ├─────────────────────────────────────────────────────────────┤
   │ ?- azione(X, Y).                                            │
   │    → Mostra TUTTE le azioni per TUTTI gli ordini            │
   │    → Premi "Next" per vedere i risultati successivi         │
   └─────────────────────────────────────────────────────────────┘

   ┌─────────────────────────────────────────────────────────────┐
   │ ALTRE QUERY UTILI                                           │
   ├─────────────────────────────────────────────────────────────┤
   │ ?- azione(ord_0, Azione).                                   │
   │    → Azione per un ordine specifico                         │
   │                                                             │
   │ ?- azione(ID, annulla_e_riprogramma).                       │
   │    → Tutti gli ordini da annullare                          │
   │                                                             │
   │ ?- azione(ID, contatta_autista).                            │
   │    → Tutti gli autisti da contattare                        │
   │                                                             │
   │ ?- consegne_da_annullare(Lista).                            │
   │    → Lista ordini da annullare                              │
   │                                                             │
   │ ?- report_consegna(ID, M, T, E, S, A).                      │
   │    → Report dettagliato completo                            │
   │                                                             │
   │ ?- conta_azioni(annulla_e_riprogramma, N).                  │
   │    → Conta quante consegne da annullare                     │
   └─────────────────────────────────────────────────────────────┘

💡 TIP: Usa "Run" per eseguire e "Next" per iterare sui risultati!

══════════════════════════════════════════════════════════════════
""")

def main():
    """Funzione principale per generare il sistema esperto."""
    print("\n" + "="*60)
    print("   GENERATORE SISTEMA ESPERTO PROLOG - Logistica Amazon")
    print("="*60 + "\n")
    
    # Configurazione percorsi
    script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
    project_dir = os.path.dirname(script_dir)
    
    # Ricerca file CSV input
    possibili_percorsi = [
        os.path.join(project_dir, 'Bayesian', 'bayesian_training_data.csv'),
        os.path.join(project_dir, 'bayesian_training_data.csv'),
        os.path.join(script_dir, 'bayesian_training_data.csv'),
        '../Bayesian/bayesian_training_data.csv',
        'bayesian_training_data.csv'
    ]
    
    df = None
    for percorso in possibili_percorsi:
        if os.path.exists(percorso):
            df = pd.read_csv(percorso, nrows=20)
            print(f"✓ Dati caricati da: {percorso}")
            break
    
    if df is None:
        print("⚠ File CSV non trovato. Generazione dati di esempio...")
        import random
        random.seed(42)
        df = pd.DataFrame({
            'Weather_conditions': [random.choice(['sunny', 'cloudy', 'fog', 'stormy', 'sandstorms', 'windy']) for _ in range(20)],
            'Road_traffic_density': [random.choice(['low', 'medium', 'high', 'jam']) for _ in range(20)],
            'Age_Category': [random.choice(['giovane', 'adulto', 'senior']) for _ in range(20)],
            'Delivery_Status': [random.choice(['puntuale', 'ritardo']) for _ in range(20)]
        })
    
    # Mostra anteprima dati
    print(f"\n📊 Anteprima dati ({len(df)} righe):")
    print(df.head(5).to_string(index=False))
    
    # Genera componenti Knowledge Base
    header = genera_header()
    fatti = genera_fatti(df)
    regole = genera_regole()
    
    # Scrivi file Prolog
    output_path = os.path.join(script_dir if os.path.isdir(script_dir) else '.', 'sistema_esperto.pl')
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(header + fatti + "\n" + regole)
    
    print(f"\n✓ Knowledge Base generata: {output_path}")
    print(f"   • Fatti: {min(len(df), 20)}")
    print(f"   • Regole: 4 (sicurezza, supporto, performance, default)")
    print(f"   • Query utilità: 5")
    
    # Istruzioni per SWISH
    stampa_istruzioni_swish()
    
    print("✅ Generazione completata!\n")

# Esecuzione
main()


   GENERATORE SISTEMA ESPERTO PROLOG - Logistica Amazon

⚠ File CSV non trovato. Generazione dati di esempio...

📊 Anteprima dati (20 righe):
Weather_conditions Road_traffic_density Age_Category Delivery_Status
             windy               medium       adulto        puntuale
             sunny                  low       senior         ritardo
             sunny               medium       adulto        puntuale
             windy                  jam      giovane        puntuale
               fog               medium       senior        puntuale

✓ Knowledge Base generata: /Users/michelerusso/Documents/Universita/Icon/MyProject/Amazon_AI/Prolog/sistema_esperto.pl
   • Fatti: 20
   • Regole: 4 (sicurezza, supporto, performance, default)
   • Query utilità: 5

╔══════════════════════════════════════════════════════════════════╗
║       ISTRUZIONI PER TESTARE SU SWISH PROLOG                    ║
╚══════════════════════════════════════════════════════════════════╝

📌 PASSO 1: Accedi a